In [5]:
import numpy as np


In [10]:
#M3
#boosting + stacking if time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd

# df = pd.read_csv(r"data\claims_train_scaled.csv")
df = pd.read_csv(r"../../data/claims_train_scaled.csv")
# df.drop("Area",axis=1,inplace=True)
# fuel_map = {'Regular': 0, 'Diesel': 1}
# df['VehGas'] = df['VehGas'].map(fuel_map)
# x = df.drop(columns=["ClaimNb","Region","VehBrand","VehGas"])
df["Claim"] = (df["ClaimNb"] > 0).astype(int)
x = df.drop(columns=["ClaimNb"])
x = pd.get_dummies(x, columns=['Region','VehGas','VehBrand'], drop_first=True, dtype=float)

y = df["ClaimNb"]

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# standard boosting
model = HistGradientBoostingRegressor(
    max_iter=1000, 
    learning_rate=0.46, 
    max_depth=7,
    categorical_features=["Region","VehBrand","VehGas"],
)
model.fit(x_train, y_train)
predictions = model.predict(x_test)
r2_score(y_test,predictions)

In [12]:
#M3
#boosting + stacking if time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score, recall_score,precision_score, confusion_matrix
import pandas as pd 

# df = pd.read_csv(r"data\claims_train_scaled.csv")
# df.drop("Area",axis=1,inplace=True)
df = pd.read_csv(r"../../data/claims_train_scaled.csv")
fuel_map = {'Regular': 0, 'Diesel': 1}
df['VehGas'] = df['VehGas'].map(fuel_map)

df["Claim"] = (df["ClaimNb"] > 0).astype(int)
# x = df.drop(columns=["ClaimNb","Region","VehBrand","Claim"])
x = df.drop(columns=["ClaimNb","Claim"])
x = pd.get_dummies(x, columns=['Region','VehGas','VehBrand'], drop_first=True, dtype=float)
y = df["Claim"]

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# standard boosting
# model = HistGradientBoostingClassifier(
#     max_iter=1000, 
#     early_stopping=True,
#     learning_rate=0.46, 
#     max_depth=7,
#     categorical_features=['Region','VehGas','VehBrand'],
# )
# model.fit(x_train, y_train)
# predictions = model.predict(x_test)
# confusion_matrix(y_test,predictions)

In [8]:
model.score(x_test,y_test)

0.9491700861335995

In [69]:
precision_score(y_test,predictions)

0.3191489361702128

In [70]:
recall_score(y_test,predictions)


0.0027573529411764708

In [14]:
#RandomSearch 
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

param_dict = {
    'max_depth': [3,4,5,6,7],
    'learning_rate' : np.arange(0.01, 0.5, 0.01)
}

dt_classifier = HistGradientBoostingClassifier(
    max_iter=1000,
    early_stopping=True,
)

dt_classifier.fit(x_train, y_train)
predictions = dt_classifier.predict(x_test)
print(confusion_matrix(y_test,predictions))
print(dt_classifier.score(x_test,y_test))
print(precision_score(y_test,predictions))
print(recall_score(y_test,predictions))

# random_search = RandomizedSearchCV(
#     estimator=dt_classifier,
#     param_distributions=param_dict,
#     n_iter=50,  # Number of parameter settings that are sampled
#     cv=5,  # Number of cross-validation folds
#     verbose=2,
#     scoring="recall",
#     random_state=42,
#     n_jobs=-1  # Use all available processors
# )


[[102763      1]
 [  5440      0]]
0.9497153524823482
0.0
0.0


In [62]:
df = pd.read_csv(r"data\claims_train_scaled.csv")
df.drop("Area",axis=1,inplace=True)
fuel_map = {'Regular': 0, 'Diesel': 1}
df['VehGas'] = df['VehGas'].map(fuel_map)

df["Claim"] = (df["ClaimNb"] > 0).astype(int)
x = df.drop(columns=["ClaimNb","Region","VehBrand","Claim"])
y = df["Claim"]

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [63]:
random_search.fit(x_train,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


,estimator,HistGradientB...max_iter=1000)
,param_distributions,"{'learning_rate': array([0.01, ..., 0.48, 0.49]), 'max_depth': [3, 4, ...]}"
,n_iter,50
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [1]:
# Retrieve the best parameters found by RandomizedSearchCV
best_params = random_search.best_params_
print("Best Parameters:", best_params)

# Evaluate the model with best parameters on the test set
best_estimator = random_search.best_estimator_
test_accuracy = best_estimator.score(x_test, y_test)
print("Test r2 with Best Parameters:", test_accuracy)

NameError: name 'random_search' is not defined

In [65]:
results_df = pd.DataFrame(random_search.cv_results_)
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_learning_rate,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,16.022151,1.617747,2.770287,0.240939,7,0.05,"{'max_depth': 7, 'learning_rate': 0.05}",0.000230,0.000230,0.000461,0.000000,0.000230,0.000230,0.000146,50
1,50.394880,9.614654,9.886649,1.970408,4,0.02,"{'max_depth': 4, 'learning_rate': 0.02}",0.000691,0.000230,0.000230,0.000230,0.000461,0.000369,0.000184,46
2,5.094888,0.760158,0.559156,0.069864,6,0.31,"{'max_depth': 6, 'learning_rate': 0.31}",0.004377,0.002995,0.001843,0.002995,0.002534,0.002949,0.000829,15
3,4.039081,1.005134,0.574287,0.127190,5,0.43,"{'max_depth': 5, 'learning_rate': 0.43}",0.003686,0.002304,0.004147,0.001382,0.004377,0.003179,0.001151,13
4,2.982497,0.392587,0.451740,0.068308,7,0.40,"{'max_depth': 7, 'learning_rate': 0.4}",0.004607,0.004377,0.003226,0.002764,0.002764,0.003548,0.000793,8
5,3.511857,0.505996,0.587961,0.101027,5,0.36,"{'max_depth': 5, 'learning_rate': 0.3600000000...",0.004607,0.004147,0.002765,0.002764,0.003225,0.003502,0.000748,9
6,2.971972,0.285901,0.468400,0.061926,6,0.39,"{'max_depth': 6, 'learning_rate': 0.39}",0.005068,0.003686,0.004839,0.002304,0.002995,0.003778,0.001057,6
7,3.913341,0.766398,0.641580,0.171004,7,0.25,"{'max_depth': 7, 'learning_rate': 0.25}",0.001382,0.001382,0.002074,0.001382,0.000691,0.001382,0.000437,31
8,37.025919,5.164362,7.062324,1.016117,7,0.02,"{'max_depth': 7, 'learning_rate': 0.02}",0.000230,0.000000,0.000230,0.000230,0.000921,0.000323,0.000312,49
9,7.392969,0.918804,1.329281,0.181222,4,0.21,"{'max_depth': 4, 'learning_rate': 0.2100000000...",0.003686,0.001382,0.001382,0.000921,0.002073,0.001889,0.000971,26
